# Bamiyan Image Scoring and Annotation Toolkit

Interactive research notebook for calibration, structured annotation, weighted scoring, ranking and export of cultural-heritage reconstruction candidates.

> **Public research version:** raw archival imagery, participant-level data and generated research outputs are not included. Configure the relative data paths before running.



# Bamiyan — Image Scoring, Calibration, and Edit‑Remix Toolkit

This notebook lets you **load 10 images**, **calibrate pixel→meter scale**, **annotate iconography/architecture** (0/1/2/N/A), **compute weighted scores & rankings**, **export CSV/JSON**, **make plots**, and **draw labeled rectangles** for targeted Sora *edit‑remix* corrections.

> **Folders**: All outputs go to `./out/`. You can change `IMG_DIR` and the image list in **Cell 2 — Config**.


## Cell 1 — Setup

In [ ]:

# Install dependencies (safe to re-run). Internet is not required.
import sys, subprocess, pkgutil

def pip_install(pkg):
    if pkg.replace("-", "_") not in {m.name for m in pkgutil.iter_modules()}:
        subprocess.check_call([sys.executable, "-m", "pip", "install", pkg])

for p in ["ipywidgets","pandas","numpy==1.26.4","Pillow","matplotlib","opencv-python","jsonschema"]:
    pip_install(p)

# Enable widgets (Notebook Classic)
try:
    import ipywidgets as widgets  # noqa
    from IPython.display import display
    _ = subprocess.run([sys.executable, "-m", "jupyter", "nbextension", "enable", "--py", "widgetsnbextension"], check=False)
except Exception as e:
    print("Widgets enable step skipped:", e)

import os, json, math, time, uuid, shutil
from pathlib import Path
import numpy as np
import pandas as pd
from PIL import Image
import matplotlib.pyplot as plt
from matplotlib.widgets import RectangleSelector

plt.rcParams["figure.dpi"] = 120
os.makedirs("out", exist_ok=True)
print("Setup complete.")


## Cell 2 — Config

In [ ]:

# >>> EDIT HERE if needed <<<
IMG_DIR = Path("./final images")  # put your 10 images here

# If running this notebook exactly as received from ChatGPT with demo images,
# uncomment the line below to point to the provided path (used in chat sandbox).
# IMG_DIR = Path("/mnt/data")  # demo path used in the chat environment

# The 10 images. You may change filenames or order. Filenames become ImageIDs.
IMAGE_FILES = [
    "1.png","2.png","3.png","4.png","5.png",
    "6.png","7.png","8.png","9.png","10.png","11.webp","12.png","13.png","14.png"
]

# World-scale constants (m)
STATUE_TOTAL_HEIGHT = 58.0
HEAD_HEIGHT = 8.0
SHOULDER_WIDTH = 17.0
NICHE = {"h": 58.0, "w_base": 21.0, "d": 18.0}

# Acceptance thresholds for WeightedScore
EXEMPLAR_THRESH = 0.85
MINOR_EDITS_THRESH = (0.75, 0.84)  # inclusive range
MAJOR_EDITS_THRESH = 0.75

# Criteria schema (0/1/2/NA) — values are stored per image.
CRITERIA = [
    "NicheDimFit","HeightFit",
    "Hairstyle","Ears","Eyes","Nose","Lips","Urna","NeckLines",
    "Drapery","Colorway","HandsPose",
    "Behind-Feet Caves","WoodenGates","AnklesExposed","Front-Feet Element"
]

# Weights (sum to 1.0). Sub-criteria split as agreed in the rubric.
WEIGHTS = {
    "NicheDimFit": 0.10,
    "HeightFit": 0.10,

    "Ears": 0.04, "Eyes": 0.04, "Nose": 0.04, "Lips": 0.04, "Urna": 0.04,  # 0.20 total

    "Hairstyle": 0.05,
    "NeckLines": 0.05,

    "Drapery": 0.12,  # folds morphology etc.
    "AnklesExposed": 0.03,

    "Colorway": 0.10,
    "HandsPose": 0.10,

    "Behind-Feet Caves": 0.07,
    "Front-Feet Element": 0.03,

    "WoodenGates": 0.05
}

assert abs(sum(WEIGHTS.values()) - 1.0) < 1e-6, "Weights must sum to 1.0"

# JSON schema for annotations
ANNOTATION_SCHEMA = {
    "type": "object",
    "properties": {
        "ImageID": {"type": "string"},
        "scores": {
            "type": "object",
            "properties": {k: {"enum": [0,1,2,"N/A"]} for k in CRITERIA},
            "additionalProperties": False
        },
        "notes": {"type": "string", "maxLength": 120},
        "timestamp": {"type": "string"}
    },
    "required": ["ImageID","scores","timestamp"]
}

display(dict(IMG_DIR=str(IMG_DIR.resolve()), IMAGES=IMAGE_FILES, WORLD_SCALE=dict(STATUE_TOTAL_HEIGHT=STATUE_TOTAL_HEIGHT, HEAD_HEIGHT=HEAD_HEIGHT, SHOULDER_WIDTH=SHOULDER_WIDTH, NICHE=NICHE)))


## Cell 3 — Image Loader & Preview

In [ ]:

from IPython.display import display, HTML

def load_images(img_dir: Path, image_files):
    imgs = {}
    for f in image_files:
        p = img_dir / f
        if p.exists():
            imgs[f] = Image.open(p).convert("RGB")
        else:
            print("WARNING: Missing image:", p)
    return imgs

IMAGES = load_images(IMG_DIR, IMAGE_FILES)
print(f"Loaded {len(IMAGES)} / {len(IMAGE_FILES)} images.")

# Show thumbnails with IDs
ncols = 5
n = len(IMAGES)
nrows = (n + ncols - 1) // ncols
plt.figure(figsize=(14, 2.8*nrows))
for i, (img_id, img) in enumerate(IMAGES.items()):
    ax = plt.subplot(nrows, ncols, i+1)
    ax.imshow(img)
    ax.set_title(img_id, fontsize=8)
    ax.axis("off")
plt.tight_layout()
plt.show()


## Cell 4 — Pixel→Meter Calibration (per image)

In [ ]:

# Click the top of the head and base of the feet to calibrate pixel→meter scale (58 m).
# Optionally click left-shoulder and right-shoulder (outer points) and the chin and top-of-head to assess shoulder width and head height vs targets.
# Results save to `out/calibration.json`.
from IPython.display import clear_output

CAL_PATH = Path("out/calibration.json")
cal_data = {}
if CAL_PATH.exists():
    cal_data = json.loads(CAL_PATH.read_text())

def calibrate_image(img_id):
    img = IMAGES[img_id]
    fig, ax = plt.subplots()
    ax.imshow(img)
    ax.set_title(f"Calibration: {img_id}\\nClick top-of-head, then base-of-feet; optional: left-shoulder, right-shoulder, chin, top-of-head again", fontsize=9)
    pts = plt.ginput(n=6, timeout=0)  # you can hit Enter to stop after first 2
    plt.close(fig)

    if len(pts) < 2:
        print("Need at least two clicks.")
        return

    # Required two points:
    (x1, y1), (x2, y2) = pts[0], pts[1]
    pix_dist = ((x2-x1)**2 + (y2-y1)**2)**0.5
    px_per_meter = pix_dist / STATUE_TOTAL_HEIGHT

    # Optional
    shoulders = head = None
    if len(pts) >= 4:
        (lx, ly), (rx, ry) = pts[2], pts[3]
        shoulders = abs(rx - lx)
    if len(pts) >= 6:
        (cx, cy), (tx, ty) = pts[4], pts[5]
        head = abs(ty - cy)

    # Errors (%)
    err = {}
    if shoulders is not None:
        err["shoulder_pct_err"] = abs((shoulders/px_per_meter) - SHOULDER_WIDTH) / SHOULDER_WIDTH * 100.0
    if head is not None:
        err["head_pct_err"] = abs((head/px_per_meter) - HEAD_HEIGHT) / HEAD_HEIGHT * 100.0

    cal_data[img_id] = dict(px_per_meter=px_per_meter, pix_dist=pix_dist, optional_errors=err, timestamp=time.strftime("%Y-%m-%dT%H:%M:%S"))
    CAL_PATH.write_text(json.dumps(cal_data, indent=2))
    print(f"Saved calibration for {img_id}. px/m={px_per_meter:.3f}. Optional errors:", err)

# Helper to run for one image at a time
print("Run: calibrate_image('1.png')  # etc.")


## Cell 5 — Annotation UI

In [ ]:

from jsonschema import validate, ValidationError
import ipywidgets as widgets
from IPython.display import display

ANN_PATH = Path("out/annotations.json")
annotations = {}
if ANN_PATH.exists():
    annotations = json.loads(ANN_PATH.read_text())

def score_dropdown(description):
    return widgets.Dropdown(
        options=[("N/A","N/A"), ("0 — incorrect/absent",0), ("1 — partial/ambiguous",1), ("2 — correct/clear",2)],
        value="N/A", description=description, layout=widgets.Layout(width="auto")
    )

def annotate_image(img_id):
    img = IMAGES[img_id]
    display(img)

    drops = {c: score_dropdown(c) for c in CRITERIA}
    notes = widgets.Textarea(description="Notes", placeholder="≤120 chars", layout=widgets.Layout(width="100%"))
    save_btn = widgets.Button(description="Save", button_style="success")
    warn = widgets.HTML(value="")

    box = widgets.VBox(list(drops.values()) + [notes, save_btn, warn])
    display(box)

    def _auto_checks(_):
        msgs = []
        # Simple auto-checks
        if drops["AnklesExposed"].value == 0 and drops["Drapery"].value == 2:
            msgs.append("⚠️ AnklesExposed=0 but Drapery=2 — please re-check consistency.")
        # Dimensional suggestion if error >3%
        err = (cal_data.get(img_id, {}).get("optional_errors") or {})
        head_err = err.get("head_pct_err", 0)
        if head_err and head_err > 3 and drops["HeightFit"].value == 2:
            msgs.append(f"⚠️ Head height error {head_err:.1f}% > 3%; consider lowering HeightFit.")
        warn.value = "<br>".join(msgs)

        rec = {
            "ImageID": img_id,
            "scores": {k: v.value for k,v in drops.items()},
            "notes": notes.value[:120],
            "timestamp": time.strftime("%Y-%m-%dT%H:%M:%S")
        }
        try:
            validate(instance=rec, schema=ANNOTATION_SCHEMA)
            annotations[img_id] = rec
            ANN_PATH.write_text(json.dumps(annotations, indent=2))
            print(f"Saved annotations for {img_id}.")
        except ValidationError as e:
            print("Validation error:", e)

    save_btn.on_click(_auto_checks)

print("Run: annotate_image('1.png')  # etc.")


## Cell 6 — Edit‑Remix Region Labeling

In [ ]:

# Draw labeled rectangles to mark regions for Sora edit‑remix:
# {left_hand, right_hand, feet, bun_separation, remove_house_left, caves_add_wooden_gates, caves_shape_variety, measurement_fix}.
# Use the widget to pick a label, then drag on the image. Repeat to add more.
# Click 'Finish' to save to `out/regions/<ImageID>.json` (normalized coords). Also writes `out/regions_todo.csv`.

REGION_DIR = Path("out/regions")
REGION_DIR.mkdir(parents=True, exist_ok=True)

LABELS = [
    "left_hand","right_hand","feet","bun_separation","remove_house_left",
    "caves_add_wooden_gates","caves_shape_variety","measurement_fix"
]

regions = {}

def label_regions(img_id):
    img = IMAGES[img_id]
    w, h = img.size
    if img_id not in regions:
        regions[img_id] = []

    fig, ax = plt.subplots()
    ax.imshow(img)
    ax.set_title(f"Label rectangles for: {img_id}", fontsize=10)

    dropdown = widgets.Dropdown(options=LABELS, value=LABELS[0], description="Label")
    display(dropdown)

    def on_select(eclick, erelease):
        x1, y1 = eclick.xdata, eclick.ydata
        x2, y2 = erelease.xdata, erelease.ydata
        if None in (x1, y1, x2, y2):
            return
        x, y = min(x1, x2), min(y1, y2)
        w_box, h_box = abs(x2-x1), abs(y2-y1)
        regions[img_id].append({
            "label": dropdown.value,
            "rect": {
                "x": float(x/w),
                "y": float(y/h),
                "w": float(w_box/w),
                "h": float(h_box/h)
            }
        })
        ax.add_patch(plt.Rectangle((x, y), w_box, h_box, fill=False))
        fig.canvas.draw_idle()

    toggle_selector = RectangleSelector(ax, on_select, drawtype='box', useblit=True,
                                        button=[1], minspanx=5, minspany=5, spancoords='data', interactive=True)
    display(widgets.HTML("Draw a rectangle with the mouse. Change 'Label' above as needed. Close the figure when done."))
    plt.show()

    # Save button
    btn = widgets.Button(description="Finish & Save", button_style="success")
    display(btn)

    def _finish(_):
        outp = REGION_DIR / f"{img_id}.json"
        outp.write_text(json.dumps(regions[img_id], indent=2))
        print("Saved:", outp)

        # write consolidated CSV
        rows = []
        for iid, rects in regions.items():
            for r in rects:
                rows.append({
                    "ImageID": iid, "Label": r["label"],
                    "x": r["rect"]["x"], "y": r["rect"]["y"],
                    "w": r["rect"]["w"], "h": r["rect"]["h"]
                })
        if rows:
            df = pd.DataFrame(rows)
            df.to_csv("out/regions_todo.csv", index=False)
            print("Updated: out/regions_todo.csv")

    btn.on_click(_finish)

print("Run: label_regions('1.png')  # etc.")


## Cell 7 — Scoring & Ranking

In [ ]:

def compute_weighted_score(scores: dict, weights: dict):
    # N/A renormalization, 2 → 1.0, 1 → 0.5, 0 → 0.0
    total_w = 0.0
    total = 0.0
    for k, w in weights.items():
        val = scores.get(k, "N/A")
        if val == "N/A":
            continue
        total_w += w
        total += (val/2.0) * w
    return (total/total_w) if total_w > 0 else np.nan, (1.0 - total/total_w) if total_w>0 else np.nan

def rank_all():
    if not annotations:
        print("No annotations found. Use Cell 5.")
        return

    rows = []
    for img_id, rec in annotations.items():
        ws, loss = compute_weighted_score(rec["scores"], WEIGHTS)
        # dominant penalties: which criteria contributed >30% of the loss
        losses = []
        total_w = sum(w for k,w in WEIGHTS.items() if rec["scores"].get(k,"N/A")!="N/A")
        for k, w in WEIGHTS.items():
            val = rec["scores"].get(k, "N/A")
            if val == "N/A": 
                continue
            crit_loss = (1 - (val/2.0)) * (w/total_w)
            losses.append((k, crit_loss))
        dom = [k for k, l in sorted(losses, key=lambda x: -x[1]) if l >= 0.30]  # >=30% of loss

        row = {"ImageID": img_id, **rec["scores"], "Notes": rec.get("notes",""), "WeightedScore": round(ws, 4)}
        row["DominantPenalties"] = ";".join(dom)
        rows.append(row)

    df = pd.DataFrame(rows).sort_values("WeightedScore", ascending=False).reset_index(drop=True)
    df["Rank"] = np.arange(1, len(df)+1)
    df.to_csv("out/table_ranked.csv", index=False)
    print("Saved: out/table_ranked.csv")
    display(df.head(10))

    # best-for-X tags
    tags = []
    if len(df):
        top3 = df.head(3)["ImageID"].tolist()
        tags.append(("Top-3", ", ".join(top3)))
        # criterion-wise best (ties allowed)
        for crit in WEIGHTS.keys():
            if crit in df.columns:
                best_val = df[crit].max()
                best_ids = df[df[crit]==best_val]["ImageID"].tolist()
                tags.append((f"best {crit}", ", ".join(best_ids)))
        pd.DataFrame(tags, columns=["Tag","ImageIDs"]).to_csv("out/best_for_tags.csv", index=False)
        print("Saved: out/best_for_tags.csv")
    return df

df_ranked = rank_all()


## Cell 8 — Plots

In [ ]:

import pandas as pd
import matplotlib.pyplot as plt

tbl = Path("out/table_ranked.csv")
if tbl.exists():
    df = pd.read_csv(tbl)
    # Histogram
    plt.figure()
    plt.hist(df["WeightedScore"].dropna(), bins=10)
    plt.xlabel("WeightedScore")
    plt.ylabel("Count")
    plt.title("Histogram of Weighted Scores")
    plt.show()

    # Bar chart: pass rates per criterion (score=2)
    crits = list(WEIGHTS.keys())
    rates = []
    for c in crits:
        if c in df.columns:
            n2 = (df[c]==2).sum()
            n = df[c].notna().sum()
            rates.append(n2 / n if n>0 else np.nan)
    plt.figure()
    plt.bar(range(len(crits)), rates)
    plt.xticks(range(len(crits)), crits, rotation=60, ha="right")
    plt.ylabel("Pass rate (score=2)")
    plt.title("Criterion Pass Rates")
    plt.tight_layout()
    plt.show()

    # Scatter: Dimensional error vs overall score (if available from calibration)
    cal = {}
    cal_path = Path("out/calibration.json")
    if cal_path.exists():
        cal = json.loads(cal_path.read_text())
    errs = []
    scs = []
    for img_id in df["ImageID"]:
        scs.append(df.loc[df["ImageID"]==img_id, "WeightedScore"].values[0])
        opt = cal.get(img_id, {}).get("optional_errors", {})
        errs.append(opt.get("head_pct_err", np.nan))
    plt.figure()
    plt.scatter(errs, scs)
    plt.xlabel("Head height error (%)")
    plt.ylabel("WeightedScore")
    plt.title("Dimensional error vs Score")
    plt.show()
else:
    print("Run Cell 7 first to create out/table_ranked.csv")


## Cell 9 — Report Assets

In [ ]:

md_out = Path("out/report_snippets.md")
if Path("out/table_ranked.csv").exists():
    df = pd.read_csv("out/table_ranked.csv")
    top3 = ", ".join(df.head(3)["ImageID"].tolist())
    md = ["# Bamiyan Results — Paste‑Ready Snippets",
          f"**Top‑3**: {top3}",
          "",
          "## Table 1 (CSV excerpt)",
          "",
          df.to_string(index=False)]
    md_out.write_text("\n".join(md))
    print("Saved:", md_out)
else:
    print("Run Cell 7 to generate rankings first.")


## Cell 10 — Safety / Integrity Checks

In [ ]:

from jsonschema import validate, ValidationError

# Validate annotation file
ok = True
ANN_PATH = Path("out/annotations.json")
if ANN_PATH.exists():
    try:
        data = json.loads(ANN_PATH.read_text())
        for k, rec in data.items():
            validate(instance=rec, schema=ANNOTATION_SCHEMA)
        print("Annotations file is valid.")
    except Exception as e:
        ok = False
        print("Annotation validation error:", e)
else:
    print("No annotations yet.")

# Confirm image count
missing = [f for f in IMAGE_FILES if f not in IMAGES]
if missing:
    ok = False
    print("Missing images:", missing)
else:
    print("All listed images loaded. Count:", len(IMAGES))

# Confirm outputs folder
print("Outputs directory:", Path("out").resolve())

print("Integrity OK?" , ok)
